## Convolution in Practice

Find out why convolutional and pooling layers are the building blocks of Convolutional Neural Networks.

We will be covering:

- Spatial dimensions

Pooling layers

When it comes to real-life applications, most images are in fact a 3D tensor with width, height, and 3 channels (R,G,B) as dimensions.

In that case, the kernel should also be a 3D tensor (k $\times$ k $\times$ $channels$). Each kernel will produce a 2D feature map. Remember the sliding happens only across width and height. We just take the dot product of all the input channels on the computation. Each kernel will produce 1 output channel.

In practice, we tend to use more than 1 kernel in order to capture different kinds of features at the same time.

![pic](https://raw.githubusercontent.com/CUTe-EmbeddedAI/images/main/images/fig17.png)

As you may have guessed, our learnable weights are now the values of our filters and can be trained with backpropagation, as usual. We can add a bias into each filter as well.

Convolutional layers can be stacked on top of others. Since convolutions are linear operators, we include non-linear activation functions in between just as we did in fully connected layers.

To recap, you have to think in terms of input channels, output channels, and kernel size. And that is exactly how we are going to define it in Pytorch.

To define a convolutional network in Pytorch, we have:

In [1]:
import torch.nn as nn

conv_layer = nn.Conv2d(in_channels=3, out_channels=5, kernel_size=5)
print(conv_layer)

Conv2d(3, 5, kernel_size=(5, 5), stride=(1, 1))


The above layer will receive an image of 3 channels (i.e., R,G,B) and will output 5 feature maps (channels) using a kernel size of 5x5x3. For simplicity, we just say 5x5 kernel.

### Spatial dimensions

Because figuring out the dimensions of each tensor can quickly become complicated, let’s examine a simple example of a convolutional layer. Assume that we have a 7x7 image and a 3x3 filter. The feature map will be of size 5x5.

![pic](https://raw.githubusercontent.com/CUTe-EmbeddedAI/images/main/images/fig18.PNG)

This is the most common approach but it is not the only one.

Sometimes, we may want to slide/move the kernel every 2 pixels instead of every 1, thus introducing an extra hyperparameter called **stride**.

In some cases, we can also pad the image around the edges with zeros in order to control the output dimensions. The amount of zero-padding on the edges introduces another parameter in the mix called **zero-padding** or simply **padding**.

If we introduce a stride of 2 and zero-padding of 1, we will receive an image of 4x4:

![pic](https://raw.githubusercontent.com/CUTe-EmbeddedAI/images/main/images/fig19.PNG)

To summarize, given an input of size $W_1 \times H_1 \times D_1$, a number of output channels $K$ with kernel size $F \times F$, stride $S$ and padding $P$, we acquire an output of size $W_2 \times H_2 \times D_2$ where:

![pic](https://raw.githubusercontent.com/CUTe-EmbeddedAI/images/main/images/fig20.PNG)

The above example can be validated in 3 lines of Pytorch code.

In the input tensor, 1 refers to the batch size. You can ignore it for now.

In [2]:
import torch
import torch.nn as nn

input_img = torch.rand(1,3,7,7)
layer = nn.Conv2d(in_channels=3, out_channels=6, kernel_size=3, stride=2, padding=1)
out = layer(input_img)
print(out.shape) 

torch.Size([1, 6, 4, 4])


### Pooling layer

Many popular CNN architectures utilize another type of layer besides the convolutional layer. This new layer is known as the pooling layer. Pooling layers can be thought of as a way to **downsample** the features without having any learnable parameters. In other words, pooling layers do not contribute to the training of a neural network. These layers function in a similar form as convolutional layers in terms that we apply a function on a chunk of the input and produce a single scalar number. The most common way is known as max-pooling and works as shown below:

![pic](https://raw.githubusercontent.com/CUTe-EmbeddedAI/images/main/images/fig21.PNG)

From each rectangular 2x2 chunk of our image, we keep only the bigger element in our feature map, resulting in a tensor with half the size of our initial input. In other words, we collapse each non-overlapping chunk into a single value.

One reason that we may introduce pooling is that it adds invariance to minor spatial changes. For example, two tensors with slightly different translations will result in the same pooling map.

Another reason is that we want to gradually reduce the resolution of the input as we perform the forward pass. That’s because the deeper layers should have a higher receptive field, meaning that they should be more and more sensitive to the entire image.

After all, our ultimate goal is to classify if an image contains a cat or a dog and not detect the corners.

Finally, pooling makes the learned features more abstract.

In [3]:
import torch
import torch.nn as nn

input_img = torch.rand(1,3,8,8)
layer = nn.MaxPool2d(kernel_size=2, stride=2)
out = layer(input_img)
print(out.shape) 

torch.Size([1, 3, 4, 4])


---
## Output Size Formula — Interactive Calculator

Given an input of size $W \times H$ with kernel $F$, padding $P$, and stride $S$:

$$W_{out} = \left\lfloor \frac{W - F + 2P}{S} \right\rfloor + 1$$

Use the function below to verify any conv or pooling layer's output size before building your network.

In [ ]:
# [UPDATED] Output size calculator — use this whenever you're unsure about dimensions
def conv_output_size(W, F, P=0, S=1):
    """Return output spatial size after Conv2d or MaxPool2d."""
    return (W - F + 2 * P) // S + 1

# --- Examples ---
print('Input 7x7, kernel 3, pad 0, stride 1 →', conv_output_size(7, 3, 0, 1))  # 5
print('Input 7x7, kernel 3, pad 1, stride 2 →', conv_output_size(7, 3, 1, 2))  # 4
print('Input 32x32, kernel 3, pad 1, stride 1 →', conv_output_size(32, 3, 1, 1))  # 32 (same-padding)
print('Input 32x32, MaxPool 2x2, stride 2 →', conv_output_size(32, 2, 0, 2))  # 16

# --- Trace a CIFAR-10 CNN forward pass ---
import torch, torch.nn as nn

class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)   # 32x32→32x32
        self.pool  = nn.MaxPool2d(2, 2)               # 32x32→16x16
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)  # 16x16→16x16
        # after pool2: 8x8, 64 channels → 64*8*8 = 4096 features
        self.fc    = nn.Linear(64 * 8 * 8, 10)

    def forward(self, x):
        x = torch.relu(self.conv1(x));  print('after conv1:', x.shape)
        x = self.pool(x);               print('after pool1:', x.shape)
        x = torch.relu(self.conv2(x));  print('after conv2:', x.shape)
        x = self.pool(x);               print('after pool2:', x.shape)
        x = x.view(x.size(0), -1);      print('after flatten:', x.shape)
        return self.fc(x)

model = SimpleCNN()
dummy = torch.zeros(1, 3, 32, 32)  # batch of 1 CIFAR-10 image
_ = model(dummy)

---
## Feature Map Visualization

After a convolution, each output channel is a **feature map** — it highlights
where the corresponding filter activates most strongly.
Visualizing feature maps helps build intuition about what early vs deep layers detect.

In [ ]:
# [UPDATED] Visualize feature maps from the first conv layer
import matplotlib.pyplot as plt
import torch, torch.nn as nn

torch.manual_seed(42)

# Create a synthetic RGB image (checkerboard pattern)
img = torch.zeros(1, 3, 32, 32)
img[0, :, ::4, :] = 1.0   # horizontal stripes in all channels
img[0, :, :, ::4] = 1.0   # vertical stripes

# Apply a single conv layer with 8 random filters
conv = nn.Conv2d(3, 8, kernel_size=3, padding=1)
with torch.no_grad():
    feat_maps = conv(img)   # shape: (1, 8, 32, 32)

fig, axes = plt.subplots(2, 5, figsize=(14, 6))
axes[0, 0].imshow(img[0].permute(1, 2, 0).clamp(0, 1))
axes[0, 0].set_title('Input Image', fontsize=10); axes[0, 0].axis('off')

for i in range(8):
    r, c = divmod(i + 1, 5)
    ax = axes[r, c]
    fmap = feat_maps[0, i].detach().numpy()
    ax.imshow(fmap, cmap='viridis')
    ax.set_title(f'Filter {i+1}', fontsize=10)
    ax.axis('off')

# Hide unused subplot
axes[1, 4].axis('off')
plt.suptitle('Feature Maps After Conv2d(3, 8, 3)', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

---
### Common Mistakes

| Mistake | Symptom | Fix |
|---------|---------|-----|
| Miscalculating flattened size for `nn.Linear` | `RuntimeError: shapes cannot be multiplied` | Use the output-size formula or add `print(x.shape)` in `forward()` |
| Using stride > 1 without updating formula | Wrong linear layer size | Always recompute $W_{out}$ when you change stride or padding |
| Pooling non-divisible sizes | Truncation (floor division) loses rows/cols | Prefer input sizes that are powers of 2 for clean downsampling |
| `in_channels` mismatch | `RuntimeError: Expected input channels` | `out_channels` of layer $l$ must equal `in_channels` of layer $l+1$ |

---
### Further Reading

- **CS231n** — Convolutional Networks: https://cs231n.github.io/convolutional-networks/  
- **PyTorch Conv2d docs**: https://pytorch.org/docs/stable/generated/torch.nn.Conv2d.html  
- **distill.pub** — *A guide to convolution arithmetic*: https://arxiv.org/abs/1603.07285

---
### Exercises

1. **Dimension arithmetic**: Given an input `(1, 1, 28, 28)` (one grayscale MNIST image),
   calculate the output size after `Conv2d(1, 16, kernel_size=5)`.
   Verify with PyTorch.

2. **Same padding**: What padding value $P$ makes `Conv2d(in, out, kernel_size=7, stride=1)`
   preserve the spatial dimensions? Verify empirically.

3. **Receptive field**: The *effective receptive field* of a stack of conv layers grows with depth.
   For two consecutive `Conv2d(kernel_size=3, padding=0, stride=1)` layers on a 9×9 input,
   what is the final output size? How many input pixels does a single output neuron 'see'?